# Compiling C code ARMv7 reference

In [1]:
%%bash
gcc -O3 -fPIC -shared -o ./armv7/libasconhash.so ./armv7/*.c

# Importing Libraries

In [7]:
import pynq
from pynq import Overlay
import pynq.lib.dma
import numpy as np
import time
import ctypes
import pandas as pd
import os

# Parameters

In [8]:
FREQ_CPU = 650e6  
FREQ_FPGA = 100e6
ITERATIONS = 1000

lib_path = os.path.abspath('./armv7/libasconhash.so')
ascon_lib = ctypes.CDLL(lib_path)

ascon_lib.crypto_hash.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_ulonglong]
ascon_lib.crypto_hash.restype = ctypes.c_int

# Loading Overlay

In [9]:
overlay = Overlay('./bitstream_files/hash256.bit')

hash_ip = overlay.crypto_hash_0

MAX_LEN = 4096
HASH_OUT_BYTES = 32 
in_buffer = pynq.allocate(shape=(MAX_LEN,), dtype=np.uint8)
out_buffer = pynq.allocate(shape=(HASH_OUT_BYTES,), dtype=np.uint8)

print("Overlay loaded")

Overlay loaded


# Test

In [10]:
test_sizes = [64, 128, 256, 512, 1024, 2048, 4096]
results = []

for size in test_sizes:
    # Generate random data
    test_data = np.random.randint(0, 256, size, dtype=np.uint8)
    
    # Prepare CPU buffers
    cpu_out = np.zeros(HASH_OUT_BYTES, dtype=np.uint8)
    
    # ---------------------------------------------------------\
    # Software test (CPU
    # ---------------------------------------------------------\
    total_time_cpu = 0.0
    for _ in range(ITERATIONS):
        start_cpu = time.perf_counter()
        ascon_lib.crypto_hash(
            cpu_out.ctypes.data_as(ctypes.c_void_p), 
            test_data.ctypes.data_as(ctypes.c_void_p), \
            ctypes.c_ulonglong(size)
        )
        end_cpu = time.perf_counter()
        total_time_cpu += (end_cpu - start_cpu)
        
    time_cpu = total_time_cpu / ITERATIONS
    cycles_cpu = time_cpu * FREQ_CPU
    
    # ---------------------------------------------------------\
    # Hardware test (FPGA)
    # ---------------------------------------------------------\
    # Copies data to the PYNQ input buffer
    np.copyto(in_buffer[:size], test_data)
    in_buffer.flush() 
    
    hash_ip.register_map.len_1 = size
    hash_ip.register_map.len_2 = 0
    hash_ip.register_map.in_r_1 = in_buffer.physical_address
    hash_ip.register_map.in_r_2 = 0
    hash_ip.register_map.out_r_1 = out_buffer.physical_address
    hash_ip.register_map.out_r_2 = 0
    
    total_time_fpga = 0.0
    for _ in range(ITERATIONS):
        start_fpga = time.perf_counter()
        
        hash_ip.register_map.CTRL.AP_START = 1
        while hash_ip.register_map.CTRL.AP_DONE == 0:
            pass
            
        end_fpga = time.perf_counter()
        total_time_fpga += (end_fpga - start_fpga)
    
    out_buffer.invalidate()
    
    time_fpga = total_time_fpga / ITERATIONS
    cycles_fpga = time_fpga * FREQ_FPGA
    
    # ---------------------------------------------------------\
    # Validation
    # ---------------------------------------------------------\
    fpga_out = np.array(out_buffer)
    is_valid = np.array_equal(cpu_out, fpga_out)
    
    results.append({
        "Size (Bytes)": size,
        "Avg Time CPU (µs)": f"{time_cpu * 1e6:.6f}",
        "Avg Cycles CPU": int(cycles_cpu),
        "Avg Time FPGA (µs)": f"{time_fpga * 1e6:.6f}",
        "Avg Cycles FPGA": int(cycles_fpga),
        "Speedup (FPGA/CPU)": f"{(time_cpu / time_fpga):.2f}x",
        "Hardware Correctness": "Pass" if is_valid else "Failed"
    })

in_buffer.freebuffer()
out_buffer.freebuffer()

df_results = pd.DataFrame(results)
display(df_results)

,Size (Bytes),Avg Time CPU (µs),Avg Cycles CPU,Avg Time FPGA (µs),Avg Cycles FPGA,Speedup (FPGA/CPU),Hardware Correctness
0,64,184.042975,119627,146.907029,14690,1.25x,Pass
1,128,188.985164,122840,137.674868,13767,1.37x,Pass
2,256,205.619990,133652,137.281456,13728,1.50x,Pass
3,512,242.682975,157743,139.622854,13962,1.74x,Pass
4,1024,310.220123,201643,138.031086,13803,2.25x,Pass
5,2048,449.384589,292099,137.843383,13784,3.26x,Pass
6,4096,727.891948,473129,199.595018,19959,3.65x,Pass
